[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drksci/phosphene/blob/main/notebooks/02_llm_labeling.ipynb)

# 02 · LLM labeling — cheapest effective path

Spend tokens only where free labels are weak:

| stage | what | cost |
|---|---|---|
| 0 | dedupe by layout signature (01) | free |
| 1 | heuristics + synthetic exact labels (01) | free |
| 2 | **select** ~1–2k frames: 60% lowest heuristic confidence + 40% k-center diversity | free |
| 3 | **label** with Claude Sonnet 5.5 · Batch API (−50%) · cached system prompt · thinking off · region (not per-cell) JSON | ≈ $0.003/frame |
| 4 | **review** with Claude Opus 5.5 only where Sonnet ≠ heuristics (top ~15%) + a 100-frame gold test set | ≈ $0.02/frame on ~15% |
| 5 | train (03), then repeat 2–4 on frames *the model* is unsure about (active learning) | — |

Expected first round for 1,500 frames: **≈ $5–10** total. A hard budget gate refuses to submit above `MAX_USD`.
Needs `ANTHROPIC_API_KEY` in Colab secrets (🔑 sidebar).

In [ ]:
#@title Setup: clone + install (re-run safe)
import os, sys
if not os.path.exists("/content/phosphene"):
    !git clone -q https://github.com/drksci/phosphene /content/phosphene
%cd /content/phosphene
!git pull -q
!pip install -q -e packages/vtcore -e packages/vttui -e "packages/vtm[train,data,label]"

#@markdown Persist data/models to Google Drive (survives runtime resets) or keep them in /content.
USE_DRIVE = False  #@param {type:"boolean"}
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA = "/content/drive/MyDrive/phosphene"
else:
    DATA = "/content/phosphene_data"
os.makedirs(DATA, exist_ok=True)
print("DATA =", DATA)

In [ ]:
#@title Parameters
N_LABEL = 1500        #@param {type:"integer"}
REVIEW_FRAC = 0.15    #@param {type:"number"}
N_GOLD = 100          #@param {type:"integer"}
MAX_USD = 20.0        #@param {type:"number"}
LABELER = "claude-sonnet-5-5"  #@param ["claude-sonnet-5-5", "claude-haiku-4-5"]
REVIEWER = "claude-opus-5-5"   #@param ["claude-opus-5-5", "claude-sonnet-5-5"]

In [ ]:
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

In [ ]:
#@title Select frames worth paying for
import glob, json, numpy as np
from vtcore import load_frames
from vtm.labeling.heuristics import frame_confidence
from vtm.labeling.select import select_frames

frames, hroles, confs = [], [], []
for p in sorted(glob.glob(f"{DATA}/weak_*.npz")):
    fr, ex = load_frames(p); frames += fr; hroles += ex["roles"]; confs += [frame_confidence(r, c.astype(np.float32)) for r, c in zip(ex["roles"], ex["conf"])]
# optional: after 03 has run, use model confidence instead (active learning round 2+)
if os.path.exists(f"{DATA}/model_conf.json"):
    confs = json.load(open(f"{DATA}/model_conf.json")); print("using model confidences")
idx = select_frames(frames, confs, N_LABEL, roles=hroles)
sel = [frames[i] for i in idx]
print(len(sel), "selected of", len(frames))

In [ ]:
#@title Cost estimate + budget gate
from vtm.labeling import llm
est = llm.estimate_cost(sel, LABELER)
est_review = llm.estimate_cost(sel[: int(len(sel) * REVIEW_FRAC) + N_GOLD], REVIEWER, out_tokens=1500)
print("labeler:", est); print("reviewer (upper bound):", est_review)
assert est["usd"] + est_review["usd"] <= MAX_USD, "over budget: lower N_LABEL"

In [ ]:
#@title Stage 3 · label with the cheap model (Batch API)
JOBS = f"{DATA}/jobs.json"
jobs = json.load(open(JOBS)) if os.path.exists(JOBS) else {}
reqs = {f"f{i}": llm.labeler_request(frames[i], LABELER) for i in idx}
if "label" not in jobs:  # resumable: re-running never double-submits
    job = llm.submit_batch(reqs, max_usd=MAX_USD, est=est)
    jobs["label"] = {"id": job.id, "ids": job.custom_ids}; json.dump(jobs, open(JOBS, "w"))
job = llm.BatchJob(jobs["label"]["id"], jobs["label"]["ids"])
llm.wait_batch(job)
labels, usage = llm.collect_batch(job)
pi, po = llm.PRICES[LABELER]
print(len(labels), "labeled", usage, f"≈ ${0.5 * (usage['input'] * pi + usage['output'] * po + usage['cache_read'] * pi * 0.1) / 1e6:.2f}")

In [ ]:
#@title Stage 4 · route disagreements (+ a gold set) to the reviewer
import random
from vtm.labeling.select import disagreement
cheap = {int(k[1:]): llm.rasterize(v["regions"], frames[int(k[1:])].shape, frames[int(k[1:])]) for k, v in labels.items()}
dis = {i: disagreement(cheap[i], hroles[i]) for i in cheap}
by_dis = sorted(dis, key=lambda i: -dis[i][0])
review_ids = by_dis[: int(len(cheap) * REVIEW_FRAC)]
rest = [i for i in cheap if i not in set(review_ids)]
gold_ids = random.Random(0).sample(rest, min(N_GOLD, len(rest)))
print("review:", len(review_ids), "gold:", len(gold_ids), "median disagreement", np.median([d for d, _ in dis.values()]).round(3))
if "review" not in jobs:
    rreqs = {f"f{i}": llm.review_request(frames[i], labels[f"f{i}"], dis[i][1], REVIEWER) for i in review_ids + gold_ids}
    job = llm.submit_batch(rreqs)
    jobs["review"] = {"id": job.id, "ids": job.custom_ids, "gold": gold_ids}; json.dump(jobs, open(JOBS, "w"))
rjob = llm.BatchJob(jobs["review"]["id"], jobs["review"]["ids"])
llm.wait_batch(rjob)
reviewed, rusage = llm.collect_batch(rjob)
print(len(reviewed), "reviewed", rusage)

In [ ]:
#@title How good is the cheap labeler? (Sonnet vs Opus on the random gold set)
from vtm.metrics import confusion, report
gold_ids = jobs["review"]["gold"]
good = [i for i in gold_ids if f"f{i}" in reviewed]
opus = {i: llm.rasterize(reviewed[f"f{i}"]["regions"], frames[i].shape, frames[i]) for i in [int(k[1:]) for k in reviewed]}
rep_cheap = report(confusion([cheap[i] for i in good], [opus[i] for i in good]))
rep_heur = report(confusion([hroles[i] for i in good], [opus[i] for i in good]))
print(f"cheap labeler vs reviewer: mIoU {rep_cheap['macro_iou']:.3f}   heuristics vs reviewer: mIoU {rep_heur['macro_iou']:.3f}")

In [ ]:
#@title Save labeled shards (gold set held out for 04)
from vtcore import save_frames
from vtcore.roles import APP_ID
def pack(ids, src):
    fs, rs, cs = [], [], []
    for i in ids:
        lab = reviewed.get(f"f{i}") or labels.get(f"f{i}")
        if lab is None: continue
        f = frames[i]; f.meta.update(source="llm", app=APP_ID.get(lab.get("app"), APP_ID["other"]), label_src=src(i))
        fs.append(f); rs.append(opus.get(i, cheap.get(i))); cs.append(np.ones(f.shape, np.float16))
    return fs, rs, cs
train_ids = [i for i in cheap if i not in set(gold_ids)]
random.Random(1).shuffle(train_ids)
n_val = max(50, len(train_ids) // 10)
src = lambda i: "reviewed" if f"f{i}" in reviewed else "labeler"
for name, ids in [("llm_val", train_ids[:n_val]), ("llm_train", train_ids[n_val:]), ("gold_test", good)]:
    fs, rs, cs = pack(ids, src); save_frames(f"{DATA}/{name}.npz", fs, {"roles": rs, "conf": cs}); print(name, len(fs))

In [ ]:
#@title Spot-check: heuristics vs cheap labeler vs reviewer
from vtm.viz import compare_html, show
for i in by_dis[:3]:
    maps = {"heuristics": hroles[i], LABELER: cheap[i]}
    if i in opus: maps[REVIEWER] = opus[i]
    show(compare_html(frames[i], maps))